# 04 - CNN Preprocessing for Google Colab

This notebook prepares the CNN training data **locally**.

We run this notebook on this machine because the raw SD302 fingerprint images are stored here. The output is a compact package that can be uploaded to Google Drive and used in Google Colab for training.

The training package contains resized grayscale image arrays, labels, split information, and metadata. It does **not** require Colab to access the raw SD302 folders.

## Important data note

The exported package is still derived from biometric fingerprint data.

Use it only for the agreed research/training purpose and keep it in your own private Google Drive. Do not publish or share it publicly.

## Install required libraries

This notebook only needs lightweight preprocessing libraries.

In [ ]:
%pip install pandas numpy pillow matplotlib tqdm

## Section 1: Set up paths and imports

This works whether the notebook is opened from the project root or from inside the `notebooks` folder.

In [ ]:
from pathlib import Path
import json
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

current_folder = Path.cwd()
PROJECT_ROOT = current_folder.parent if current_folder.name == "notebooks" else current_folder

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
COLAB_DIR = PROCESSED_DIR / "colab_package"
FIGURE_DIR = PROCESSED_DIR / "figures"

COLAB_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

COLAB_DIR

## Section 2: Load the roll-only split

This table tells us which images belong to train, validation, and test.

It also contains the target class label for each image.

In [ ]:
split_path = PROCESSED_DIR / "roll_broad_model_split.csv"

id_columns = {
    "subject_id": "string",
    "finger_position": "string",
    "resolution": "string",
}

dataset = pd.read_csv(split_path, dtype=id_columns)
dataset["image_path"] = dataset["png_path"].apply(lambda path: PROJECT_ROOT / path)

print("Rows:", len(dataset))
print("Subjects:", dataset["subject_id"].nunique())

In [ ]:
pd.crosstab(dataset["broad_class"], dataset["split"])[["train", "validation", "test"]]

## Section 3: Confirm all images exist

Colab will not use the raw images directly, but this local preprocessing step must be able to read them.

In [ ]:
dataset["image_exists"] = dataset["image_path"].apply(lambda path: path.exists())

if not dataset["image_exists"].all():
    missing_count = (dataset["image_exists"] == False).sum()
    raise FileNotFoundError(f"Missing image files: {missing_count}")

print("All image files were found.")

## Section 4: Create label mapping

Machine learning models use numeric labels, so we map each class name to an integer.

The mapping is saved so Colab can interpret model predictions correctly.

In [ ]:
label_names = sorted(dataset["broad_class"].unique())
label_to_id = {label: index for index, label in enumerate(label_names)}

dataset["label_id"] = dataset["broad_class"].map(label_to_id)

label_to_id

In [ ]:
label_mapping_path = COLAB_DIR / "label_mapping.json"

with open(label_mapping_path, "w", encoding="utf-8") as file:
    json.dump(label_to_id, file, indent=2)

print("Saved:", label_mapping_path)

## Section 5: Resize images into arrays

The CNN training notebook will load images from one compressed file.

Here we convert each fingerprint into a 160 x 160 grayscale image and store it as a compact `uint8` array.

In [ ]:
IMAGE_SIZE = 160

def preprocess_fingerprint(image_path):
    image = Image.open(image_path).convert("L")
    image = image.resize((IMAGE_SIZE, IMAGE_SIZE))
    return np.array(image, dtype=np.uint8)

In [ ]:
image_arrays = []

for image_path in tqdm(dataset["image_path"], desc="Preprocessing images"):
    image_arrays.append(preprocess_fingerprint(image_path))

images = np.stack(image_arrays)

print("Image array shape:", images.shape)
print("Data type:", images.dtype)

## Section 6: Save the Colab-ready files

We save three files:

- `roll_cnn_160x160_dataset.npz`: image arrays, labels, splits, and label names.
- `roll_cnn_metadata.csv`: readable metadata for each row.
- `label_mapping.json`: class-name to numeric-label mapping.

In [ ]:
metadata_columns = [
    "subject_id",
    "finger_position",
    "capture_type",
    "collection_type",
    "primary_label",
    "broad_class",
    "label_id",
    "split",
    "png_path",
]

metadata = dataset[metadata_columns].copy()
metadata_path = COLAB_DIR / "roll_cnn_metadata.csv"

metadata.to_csv(metadata_path, index=False)

print("Saved:", metadata_path)

In [ ]:
dataset_path = COLAB_DIR / "roll_cnn_160x160_dataset.npz"

np.savez_compressed(
    dataset_path,
    images=images,
    labels=metadata["label_id"].to_numpy(dtype=np.int64),
    splits=metadata["split"].to_numpy(),
    label_names=np.array(label_names),
)

print("Saved:", dataset_path)

## Section 7: Check the saved package

Before moving to Google Drive, we reload the saved array file and confirm the shapes are correct.

In [ ]:
saved_data = np.load(dataset_path, allow_pickle=True)

print("Images:", saved_data["images"].shape)
print("Labels:", saved_data["labels"].shape)
print("Splits:", saved_data["splits"].shape)
print("Label names:", saved_data["label_names"].tolist())

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(9, 7))

for axis, row_index in zip(axes.ravel(), range(12)):
    axis.imshow(images[row_index], cmap="gray")
    axis.set_title(metadata.loc[row_index, "broad_class"])
    axis.axis("off")

plt.tight_layout()
plt.savefig(FIGURE_DIR / "colab_preprocessed_image_preview.png", dpi=150)
plt.show()

## Section 8: Create a zip file for Google Drive

This zip file is what you upload or copy to Google Drive.

The Colab training notebook will unzip it and train from these prepared files.

In [ ]:
readme_path = COLAB_DIR / "README_colab_package.txt"

readme_text = """
CNN training package for the dermatoglyphic pattern recognition project.

Files:
- roll_cnn_160x160_dataset.npz: preprocessed fingerprint image arrays and labels.
- roll_cnn_metadata.csv: metadata for each image row.
- label_mapping.json: class-name to label-id mapping.

Data note: this package is derived from NIST SD302 biometric data. Keep it private and use it only for the agreed research/training purpose.
""".strip()

readme_path.write_text(readme_text, encoding="utf-8")

print("Saved:", readme_path)

In [ ]:
zip_path = PROCESSED_DIR / "roll_cnn_colab_package.zip"

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zip_file:
    for file_path in [dataset_path, metadata_path, label_mapping_path, readme_path]:
        zip_file.write(file_path, arcname=file_path.name)

print("Saved:", zip_path)
print("Size MB:", round(zip_path.stat().st_size / (1024 * 1024), 2))

## Stop before Google Colab

Pause here after the zip file is created.

Upload or copy this file to your private Google Drive:

`data/processed/roll_cnn_colab_package.zip`

Suggested Google Drive folder:

`MyDrive/dermatoglyphic_project/roll_cnn_colab_package.zip`

After that, open the Colab training notebook and update the package path if needed.